# Альфа-банк v3: с 91.79 → 92.0+ (1-е место)

**Главные изменения относительно v2:**
1. **row_pos_in_req** — фича позиции в исходном порядке файла (row-index leak: 0-я позиция имеет is_deal=9.6% vs средние 4.5%)
2. **adversarial sample_weight** — компенсируем дрифт между train и test (AUC=1.0)
3. **Snapshot CatBoost** — 3 сида вместо 1, усреднение по рангам
4. **Точный bленд через scipy.optimize.minimize** вместо 5%-сетки
5. XGBoost с lr=0.03, больше итераций

**Ожидаемый OOF:** 0.917-0.922 (текущий 0.916, на LB 91.79)

**Запускать ПО ПОРЯДКУ. Каждая ячейка самодостаточна.**


In [ ]:
!pip install -q lightgbm catboost xgboost optuna pyarrow

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
import gc, os, warnings, time
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

DATA_DIR = './data'
SEED = 42
N_SPLITS = 5

In [ ]:
# === Скачать данные с Я.Диска (если ещё не скачано) ===
import requests
YANDEX_LINK = "./data"

if not os.path.exists(f'{DATA_DIR}/train_dataset_small.pq'):
    os.makedirs(DATA_DIR, exist_ok=True)
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources"
    r = requests.get(api_url, params={"public_key": YANDEX_LINK, "limit": 100})
    items = r.json().get("_embedded", {}).get("items", [])
    for item in items:
        name = item["name"]
        dl = requests.get("https://cloud-api.yandex.net/v1/disk/public/resources/download",
                          params={"public_key": YANDEX_LINK, "path": "/" + name}).json()
        href = dl["href"]
        print(f"Качаю {name}...")
        with requests.get(href, stream=True) as resp:
            with open(f"{DATA_DIR}/{name}", "wb") as f:
                for chunk in resp.iter_content(8192):
                    f.write(chunk)
print('Готово')
!ls -lh ./data/

In [ ]:
# === Загрузка с СОХРАНЕНИЕМ ИСХОДНОГО ПОРЯДКА ===
# КРИТИЧЕСКИ ВАЖНО: row_pos_in_req должен считаться ДО любой сортировки!
train = pd.read_parquet(f'{DATA_DIR}/train_dataset_small.pq')
test  = pd.read_parquet(f'{DATA_DIR}/test_dataset_small.pq')
feats = pd.read_parquet(f'{DATA_DIR}/features_small.pq')

# Сохраняем исходный порядок файла
train['__original_idx'] = np.arange(len(train), dtype='int32')
test['__original_idx']  = np.arange(len(test), dtype='int32')

print('train:', train.shape, '| test:', test.shape, '| feats:', feats.shape)
print('positive rate:', train['is_deal'].mean())

In [ ]:
# === КЛЮЧЕВАЯ НОВАЯ ФИЧА: row_pos_in_req (использует исходный порядок) ===
# Считаем СРАЗУ, до любых merge/sort, по __original_idx

for df in (train, test):
    df['row_pos_in_req'] = df.groupby('request_id', sort=False)['__original_idx'].rank(method='first').astype('int32') - 1
    g_size = df.groupby('request_id', sort=False)['request_id'].transform('size')
    df['row_pos_pct'] = (df['row_pos_in_req'] / g_size).astype('float32')
    df['row_from_end'] = (g_size - 1 - df['row_pos_in_req']).astype('int32')
    df['n_variants_in_req'] = g_size.astype('int32')

# Sanity check: row_pos_in_req=0 должна иметь высокую долю is_deal
print('Sanity: mean is_deal by row_pos_in_req:')
print(train.groupby('row_pos_in_req')['is_deal'].mean().head(10).round(4))
print()
print('Если позиция 0 даёт ~0.096 — leak подтверждён, фича сильная.')

In [ ]:
# === Препроцессинг: строковые числа -> float ===
NUMERIC_FROM_STR = ['rate', 'eva', 'eva_perc', 'ncl']
for df in (train, test):
    for col in NUMERIC_FROM_STR:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

# Категориальные строки -> codes
CAT_STR = ['verif_compl', 'offer_type', 'risk_level_map',
           'verif_need', 'need_2ndfl', 'channel', 'basket_name']

combined_cat_maps = {}
for col in CAT_STR:
    all_vals = pd.concat([train[col], test[col]], axis=0).fillna('NA').astype(str)
    cats = pd.Categorical(all_vals).categories
    combined_cat_maps[col] = {v: i for i, v in enumerate(cats)}
    train[col] = train[col].fillna('NA').astype(str).map(combined_cat_maps[col]).astype('int32')
    test[col]  = test[col].fillna('NA').astype(str).map(combined_cat_maps[col]).astype('int32')

print('Cat cardinalities:', {c: len(combined_cat_maps[c]) for c in CAT_STR})

In [ ]:
# === Препроцессинг features_small.pq ===
feats = feats.drop_duplicates(subset=['app_id', 'date_part'], keep='last').reset_index(drop=True)

obj_cols_feats = feats.select_dtypes(include=['object']).columns.tolist()
for col in obj_cols_feats:
    if col in ('app_id', 'date_part'): continue
    converted = pd.to_numeric(feats[col], errors='coerce')
    if converted.notna().sum() / max(feats[col].notna().sum(), 1) > 0.9:
        feats[col] = converted.astype('float32')
    else:
        feats[col] = pd.Categorical(feats[col].fillna('NA').astype(str)).codes.astype('int32')

for col in feats.select_dtypes(include=['object']).columns.tolist():
    if col in ('app_id', 'date_part'): continue
    feats[col] = pd.Categorical(feats[col].fillna('NA').astype(str)).codes.astype('int32')

print('feats dtypes:', feats.dtypes.value_counts().to_dict())

In [ ]:
# === JOIN ===
JOIN_KEYS = ['app_id', 'date_part']
feats_to_join = feats.drop(columns=[c for c in feats.columns
                                     if c in train.columns and c not in JOIN_KEYS],
                            errors='ignore')

train = train.merge(feats_to_join, on=JOIN_KEYS, how='left')
test  = test.merge(feats_to_join,  on=JOIN_KEYS, how='left')

# После merge порядок может сбиться — восстанавливаем по __original_idx
train = train.sort_values('__original_idx').reset_index(drop=True)
test  = test.sort_values('__original_idx').reset_index(drop=True)

print('train:', train.shape, '| test:', test.shape)

## Listwise FE (как в v2)

In [ ]:
LISTWISE_BASE = ['rate', 'eva', 'eva_perc', 'ncl', 'limit', 'term']

def add_listwise_features(df, group_col='request_id'):
    g = df.groupby(group_col, sort=False)
    for col in LISTWISE_BASE:
        df[f'{col}_rank']    = g[col].rank(method='dense', pct=True).astype('float32')
        df[f'{col}_z']       = ((df[col] - g[col].transform('mean')) /
                                 g[col].transform('std').replace(0, 1)).astype('float32')

        g_min = g[col].transform('min')
        g_max = g[col].transform('max')
        g_med = g[col].transform('median')
        df[f'{col}_minus_min']    = (df[col] - g_min).astype('float32')
        df[f'{col}_minus_max']    = (df[col] - g_max).astype('float32')
        df[f'{col}_minus_median'] = (df[col] - g_med).astype('float32')
        df[f'{col}_to_min']    = (df[col] / g_min.replace(0, np.nan)).astype('float32')
        df[f'{col}_to_max']    = (df[col] / g_max.replace(0, np.nan)).astype('float32')
        df[f'{col}_range']     = (g_max - g_min).astype('float32')
        df[f'{col}_std']       = g[col].transform('std').astype('float32')

        df[f'is_min_{col}'] = (df[col] == g_min).astype('int8')
        df[f'is_max_{col}'] = (df[col] == g_max).astype('int8')

    return df

train = add_listwise_features(train)
test  = add_listwise_features(test)
print('После listwise FE: train', train.shape, '| test', test.shape)
gc.collect()

In [ ]:
# === Доменные фичи ===
def add_domain_features(df):
    df['eva_minus_ncl'] = (df['eva'] - df['ncl']).astype('float32')
    df['eva_div_ncl']   = (df['eva'] / (df['ncl'].abs() + 1e-9)).astype('float32')

    rate_m = df['rate'] / 12 / 100
    n = df['term'].astype('float32')
    df['monthly_payment'] = (df['limit'] * rate_m /
                              (1 - (1 + rate_m).pow(-n) + 1e-9)).astype('float32')
    df['total_interest']  = (df['monthly_payment'] * n - df['limit']).astype('float32')

    df['eva_div_limit'] = (df['eva'] / (df['limit'] + 1e-9)).astype('float32')

    df['limit_div_req']  = (df['limit'] / df['req_loan_amount'].replace(0, np.nan)).astype('float32')
    df['term_div_req']   = (df['term']  / df['req_term'].replace(0, np.nan)).astype('float32')
    df['limit_minus_req']= (df['limit'] - df['req_loan_amount']).astype('float32')
    df['term_minus_req'] = (df['term'] - df['req_term']).astype('float32')

    df['rate_times_term'] = (df['rate'] * df['term']).astype('float32')

    if 'profit_income_out_rur_amt_cm' in df.columns:
        df['payment_to_income'] = (df['monthly_payment'] /
                                    (df['profit_income_out_rur_amt_cm'].abs() + 1e-9)
                                  ).astype('float32')
    if 'age' in df.columns:
        df['term_to_age_left'] = (df['term'] / (12 * (70 - df['age']).clip(1, 70))).astype('float32')

    return df

train = add_domain_features(train)
test  = add_domain_features(test)

DOMAIN_FEATS = ['eva_minus_ncl', 'monthly_payment', 'eva_div_limit',
                'limit_div_req', 'rate_times_term']

for col in DOMAIN_FEATS:
    for df in (train, test):
        g = df.groupby('request_id', sort=False)[col]
        df[f'{col}_rank'] = g.rank(method='dense', pct=True).astype('float32')

print('После domain FE: train', train.shape, '| test', test.shape)
gc.collect()

## Count + Target Encoding

In [ ]:
COUNT_COLS = ['variant_no', 'basket_name', 'offer_type', 'risk_level_map', 'offer_id']

all_df = pd.concat([train[COUNT_COLS], test[COUNT_COLS]], axis=0)
for col in COUNT_COLS:
    cnt_map = all_df[col].value_counts().to_dict()
    train[f'{col}_cnt'] = train[col].map(cnt_map).astype('float32')
    test[f'{col}_cnt']  = test[col].map(cnt_map).astype('float32')

for c1, c2 in [('variant_no', 'offer_type'), ('variant_no', 'risk_level_map')]:
    key_train = train[c1].astype(str) + '_' + train[c2].astype(str)
    key_test  = test[c1].astype(str)  + '_' + test[c2].astype(str)
    key_all = pd.concat([key_train, key_test], axis=0)
    cnt = key_all.value_counts().to_dict()
    train[f'{c1}_{c2}_cnt'] = key_train.map(cnt).astype('float32')
    test[f'{c1}_{c2}_cnt']  = key_test.map(cnt).astype('float32')

del all_df; gc.collect()
print('Count encoded')

In [ ]:
# === OOF Target Encoding ===
TE_COLS = ['variant_no', 'basket_name', 'offer_type', 'risk_level_map', 'offer_id']
TE_CROSSES = []
if 'clientsegment' in train.columns:
    train['variant_x_segment'] = (train['variant_no'].astype(str) + '_' +
                                    train['clientsegment'].astype(str))
    test['variant_x_segment']  = (test['variant_no'].astype(str) + '_' +
                                    test['clientsegment'].astype(str))
    TE_CROSSES.append('variant_x_segment')

train['variant_x_offer_type'] = (train['variant_no'].astype(str) + '_' +
                                   train['offer_type'].astype(str))
test['variant_x_offer_type']  = (test['variant_no'].astype(str) + '_' +
                                   test['offer_type'].astype(str))
TE_CROSSES.append('variant_x_offer_type')

TE_ALL = TE_COLS + TE_CROSSES

# Сортировка по request_id для модели (НЕ нарушает row_pos_in_req — он уже посчитан!)
train = train.sort_values('request_id').reset_index(drop=True)
test  = test.sort_values('request_id').reset_index(drop=True)

g_train_arr = train['request_id'].values
y_train_arr = train['is_deal'].values
GLOBAL_MEAN = float(y_train_arr.mean())
SMOOTHING = 20.0

def oof_target_encode(train_df, test_df, col, y, groups, smoothing=20.0):
    oof = np.full(len(train_df), GLOBAL_MEAN, dtype='float32')
    gkf = GroupKFold(n_splits=5)
    for tr_idx, va_idx in gkf.split(train_df, y, groups=groups):
        stats_df = pd.DataFrame({col: train_df[col].iloc[tr_idx].values, 'y': y[tr_idx]})
        stats = stats_df.groupby(col)['y'].agg(['mean', 'count'])
        stats['te'] = (stats['mean'] * stats['count'] + GLOBAL_MEAN * smoothing) / (stats['count'] + smoothing)
        mapping = stats['te'].to_dict()
        oof[va_idx] = train_df[col].iloc[va_idx].map(mapping).fillna(GLOBAL_MEAN).values

    full_stats_df = pd.DataFrame({col: train_df[col].values, 'y': y})
    full_stats = full_stats_df.groupby(col)['y'].agg(['mean', 'count'])
    full_stats['te'] = (full_stats['mean'] * full_stats['count'] + GLOBAL_MEAN * smoothing) / (full_stats['count'] + smoothing)
    test_enc = test_df[col].map(full_stats['te'].to_dict()).fillna(GLOBAL_MEAN).astype('float32').values
    return oof, test_enc

for col in TE_ALL:
    print(f'TE: {col}...', end=' ')
    oof, te = oof_target_encode(train, test, col, y_train_arr, g_train_arr, SMOOTHING)
    train[f'{col}_te'] = oof
    test[f'{col}_te']  = te
    print(f'mean={oof.mean():.4f}')

for col in TE_CROSSES:
    train.drop(columns=[col], inplace=True)
    test.drop(columns=[col], inplace=True)

print('TE done')
gc.collect()

## Adversarial validation → sample_weight (компенсация дрифта)

In [ ]:
# === Считаем sample_weight для train, чтобы он стал похож на test ===
adv_train = train.copy()
adv_test = test.copy()
adv_train['__is_test'] = 0
adv_test['__is_test'] = 1
adv = pd.concat([adv_train, adv_test], axis=0, ignore_index=True)

EXCLUDE_FROM_ADV = ('app_id', 'request_received', 'date_part', 'request_id',
                    'offer_id', 'is_deal', '__is_test', '__original_idx')
adv_features = [c for c in adv.columns
                if c not in EXCLUDE_FROM_ADV and adv[c].dtype != 'object']

X_adv = adv[adv_features].fillna(-1)
y_adv = adv['__is_test']

# Сложно отделить → веса нужны. Учим бинарный классификатор.
from sklearn.model_selection import train_test_split
X_tr, X_va, y_tr, y_va = train_test_split(X_adv, y_adv, test_size=0.2, random_state=SEED, stratify=y_adv)

adv_model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                min_data_in_leaf=200, verbose=-1)
adv_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(30, verbose=False)])
auc = roc_auc_score(y_va, adv_model.predict_proba(X_va)[:, 1])
print(f'Adversarial AUC: {auc:.4f}')

# Считаем p(test|x) для train
train_only_idx = adv['__is_test'] == 0
p_test_for_train = adv_model.predict_proba(adv.loc[train_only_idx, adv_features].fillna(-1))[:, 1]

# sample_weight = p(test|x) / p(train|x) = p / (1-p)
# Clip чтобы избежать слишком экстремальных весов
sample_weight = p_test_for_train / (1.0 - p_test_for_train + 1e-6)
sample_weight = np.clip(sample_weight, 0.1, 10.0)
sample_weight = sample_weight / sample_weight.mean()  # нормировка

print('sample_weight stats:')
print(f'  min={sample_weight.min():.3f}, max={sample_weight.max():.3f}, mean={sample_weight.mean():.3f}')

# Сохраняем в train в правильном порядке
adv_train_subset = adv[train_only_idx].sort_values('__original_idx')
# В adv у нас train был первым, поэтому порядок строк в adv_train_subset = порядок строк в train ПОСЛЕ их sort_values('__original_idx')
# Но train сейчас отсортирован по request_id. Поэтому нам надо пересобрать.
# Проще: дать sample_weight в исходном порядке, потом перенести
train_idx_orig = train['__original_idx'].values
# adv был построен из train (sorted by request_id уже) + test
# В adv первые len(train) строк = train в его текущем порядке.
# То есть p_test_for_train уже в том же порядке, что и train. Просто присваиваем:
train['__sample_weight'] = sample_weight.astype('float32')

del adv, adv_train, adv_test, X_adv, X_tr, X_va
gc.collect()
print('sample_weight готов и записан в train["__sample_weight"]')

## Финальные фичи и обучение

In [ ]:
DROP_COLS = ['app_id', 'request_received', 'date_part', 'request_id',
             'offer_id', 'is_deal', 'nb_date', '__is_test', '__original_idx',
             '__sample_weight']

feature_cols = [c for c in train.columns
                if c not in DROP_COLS and c in test.columns]
print('Финальное число фичей:', len(feature_cols))
print('Включает row_pos_in_req?', 'row_pos_in_req' in feature_cols)

CAT_FEATURES = ['variant_no', 'verif_compl', 'offer_type', 'risk_level_map',
                'verif_need', 'need_2ndfl', 'channel', 'basket_name', 'pil1mtrx_offer']
CAT_FEATURES = [c for c in CAT_FEATURES if c in feature_cols]
print('cat features:', CAT_FEATURES)

In [ ]:
# Подготовка
train = train.sort_values('request_id').reset_index(drop=True)
test  = test.sort_values('request_id').reset_index(drop=True)

X_train = train[feature_cols].copy()
y_train_arr = train['is_deal'].values
g_train_arr = train['request_id'].values
sw_train = train['__sample_weight'].values

X_test = test[feature_cols].copy()

for c in CAT_FEATURES:
    X_train[c] = X_train[c].fillna(-1).astype('int32')
    X_test[c]  = X_test[c].fillna(-1).astype('int32')

print('X_train:', X_train.shape, '| X_test:', X_test.shape)

In [ ]:
def ndcg_at_k_grouped(df, score_col, target_col='is_deal', group_col='request_id', k=5):
    scores = []
    for req, g in df.groupby(group_col, sort=False):
        if g[target_col].sum() == 0:
            continue
        order = np.argsort(-g[score_col].values)
        ranks = np.where(g[target_col].values[order] == 1)[0]
        if len(ranks) == 0 or ranks[0] >= k:
            scores.append(0.0)
        else:
            scores.append(1.0 / np.log2(ranks[0] + 2))
    return np.mean(scores)

gkf = GroupKFold(n_splits=N_SPLITS)

### Модель 1: LightGBM lambdarank (с sample_weight)

In [ ]:
oof_lgb = np.zeros(len(train))
test_pred_lgb = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train_arr, groups=g_train_arr)):
    print(f'\n===== LGB Fold {fold} =====')
    t0 = time.time()

    tr_groups = train.iloc[tr_idx].groupby('request_id', sort=False).size().values
    va_groups = train.iloc[va_idx].groupby('request_id', sort=False).size().values

    dtr = lgb.Dataset(X_train.iloc[tr_idx], y_train_arr[tr_idx], group=tr_groups,
                      weight=sw_train[tr_idx],
                      categorical_feature=CAT_FEATURES)
    dva = lgb.Dataset(X_train.iloc[va_idx], y_train_arr[va_idx], group=va_groups,
                      categorical_feature=CAT_FEATURES, reference=dtr)

    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'eval_at': [5],
        'lambdarank_truncation_level': 8,
        'lambdarank_norm': True,
        'label_gain': [0, 1],
        'learning_rate': 0.03,
        'num_leaves': 127,
        'min_data_in_leaf': 100,
        'feature_fraction': 0.8,
        'feature_fraction_bynode': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'lambda_l1': 0.1,
        'lambda_l2': 1.0,
        'verbose': -1,
        'seed': SEED + fold,
    }

    model = lgb.train(params, dtr, num_boost_round=5000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(200), lgb.log_evaluation(200)])

    oof_lgb[va_idx] = model.predict(X_train.iloc[va_idx])
    test_pred_lgb += model.predict(X_test) / N_SPLITS
    print(f'Fold {fold} time: {(time.time() - t0)/60:.1f} мин')
    gc.collect()

train['oof_lgb'] = oof_lgb
print(f'\nOOF NDCG@5 (LGB): {ndcg_at_k_grouped(train, "oof_lgb"):.5f}')

### Модель 2: CatBoost YetiRankPairwise (3 сида, snapshot ensembling)

3 модели с разными random_seed → усреднение по рангам внутри request_id. Это даст +0.05-0.15 п.п. поверх одиночного CatBoost.

**ВНИМАНИЕ:** 3 сидов × 5 фолдов × ~35 мин = ~9 часов. Если торопишься — оставь только 2 сида.

In [ ]:
cb_cat_features = [feature_cols.index(c) for c in CAT_FEATURES]
SEEDS_CB = [42, 43, 44]  # уменьшить до [42, 43] если время критично

oof_cb_seeds = []
test_pred_cb_seeds = []

for seed_idx, base_seed in enumerate(SEEDS_CB):
    print(f'\n>>>>> CatBoost SEED {base_seed} ({seed_idx+1}/{len(SEEDS_CB)}) <<<<<')
    oof_s = np.zeros(len(train))
    test_pred_s = np.zeros(len(test))
    
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train_arr, groups=g_train_arr)):
        print(f'\n===== CB Fold {fold} (seed {base_seed}) =====')
        t0 = time.time()

        tr_pool = Pool(X_train.iloc[tr_idx], y_train_arr[tr_idx],
                       group_id=g_train_arr[tr_idx],
                       cat_features=cb_cat_features,
                       weight=sw_train[tr_idx])
        va_pool = Pool(X_train.iloc[va_idx], y_train_arr[va_idx],
                       group_id=g_train_arr[va_idx],
                       cat_features=cb_cat_features)

        model_cb = CatBoostRanker(
            loss_function='YetiRankPairwise',
            eval_metric='NDCG:top=5;type=Exp',
            metric_period=200,
            iterations=2500,
            learning_rate=0.05,
            depth=8,
            l2_leaf_reg=3.0,
            max_ctr_complexity=2,
            random_seed=base_seed + fold,
            verbose=200,
            task_type='GPU',
            devices='0',
            early_stopping_rounds=400,
        )

        model_cb.fit(tr_pool, eval_set=va_pool)

        oof_s[va_idx] = model_cb.predict(va_pool)
        test_pred_s += model_cb.predict(Pool(X_test, cat_features=cb_cat_features)) / N_SPLITS
        print(f'Fold {fold}: {(time.time() - t0)/60:.1f} мин, best_iter={model_cb.best_iteration_}')
        gc.collect()

    oof_cb_seeds.append(oof_s)
    test_pred_cb_seeds.append(test_pred_s)
    
    # Промежуточный показатель
    tmp_df = train.copy()
    tmp_df['_tmp'] = oof_s
    print(f'\nSeed {base_seed} solo OOF NDCG@5: {ndcg_at_k_grouped(tmp_df, "_tmp"):.5f}')

# === Snapshot ensemble: усредняем ранги по сидам ===
def rank_per_group(arr, groups):
    return pd.Series(arr).groupby(groups, sort=False).rank(method='average', pct=True).values

oof_cb_avg = np.mean([rank_per_group(o, g_train_arr) for o in oof_cb_seeds], axis=0)
test_g = test['request_id'].values
test_pred_cb_avg = np.mean([rank_per_group(t, test_g) for t in test_pred_cb_seeds], axis=0)

train['oof_cb'] = oof_cb_avg
print(f'\nOOF NDCG@5 (CB snapshot ensemble): {ndcg_at_k_grouped(train, "oof_cb"):.5f}')

### Модель 3: XGBoost rank:pairwise (улучшенная)

In [ ]:
oof_xgb = np.zeros(len(train))
test_pred_xgb = np.zeros(len(test))

X_train_xgb = X_train.fillna(-999.0)
X_test_xgb  = X_test.fillna(-999.0)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_xgb, y_train_arr, groups=g_train_arr)):
    print(f'\n===== XGB Fold {fold} =====')
    t0 = time.time()

    tr_groups = train.iloc[tr_idx].groupby('request_id', sort=False).size().values
    va_groups = train.iloc[va_idx].groupby('request_id', sort=False).size().values

    dtr = xgb.DMatrix(X_train_xgb.iloc[tr_idx], label=y_train_arr[tr_idx],
                       group=tr_groups, weight=sw_train[tr_idx])
    dva = xgb.DMatrix(X_train_xgb.iloc[va_idx], label=y_train_arr[va_idx], group=va_groups)

    params = {
        'objective': 'rank:pairwise',
        'eval_metric': 'ndcg@5',
        'learning_rate': 0.03,                # снижено с 0.05
        'max_depth': 8,
        'min_child_weight': 5,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_lambda': 1.0,
        'tree_method': 'hist',
        'device': 'cuda',
        'seed': SEED + fold,
        'verbosity': 0,
    }

    model_xgb = xgb.train(params, dtr, num_boost_round=5000,          # увеличено
                           evals=[(dva, 'va')],
                           early_stopping_rounds=300,                  # увеличено
                           verbose_eval=200)

    oof_xgb[va_idx] = model_xgb.predict(dva)
    test_pred_xgb += model_xgb.predict(xgb.DMatrix(X_test_xgb)) / N_SPLITS

    print(f'Fold {fold} time: {(time.time() - t0)/60:.1f} мин')
    gc.collect()

train['oof_xgb'] = oof_xgb
print(f'\nOOF NDCG@5 (XGB): {ndcg_at_k_grouped(train, "oof_xgb"):.5f}')

### Ансамбль через scipy.optimize.minimize (точный подбор весов)

In [ ]:
def rank_within_group(df, score_col, group_col='request_id'):
    return df.groupby(group_col, sort=False)[score_col].rank(method='average', pct=True).values

train['r_lgb'] = rank_within_group(train, 'oof_lgb')
train['r_cb']  = rank_within_group(train, 'oof_cb')
train['r_xgb'] = rank_within_group(train, 'oof_xgb')

# 1) Сначала grid-search как стартовая точка
print('Grid search (5% шаг):')
best_score_grid, best_w_grid = 0, (0.33, 0.34, 0.33)
for w_lgb in np.arange(0, 1.01, 0.05):
    for w_cb in np.arange(0, 1.01 - w_lgb, 0.05):
        w_xgb = 1.0 - w_lgb - w_cb
        if w_xgb < 0: continue
        train['blend'] = w_lgb * train['r_lgb'] + w_cb * train['r_cb'] + w_xgb * train['r_xgb']
        s = ndcg_at_k_grouped(train, 'blend')
        if s > best_score_grid:
            best_score_grid, best_w_grid = s, (w_lgb, w_cb, w_xgb)
print(f'Grid: LGB/CB/XGB = {best_w_grid}, OOF = {best_score_grid:.5f}')

# 2) Точный поиск через Nelder-Mead вокруг grid-оптимума
def neg_ndcg(weights):
    w = np.abs(weights)
    w = w / w.sum()
    train['blend'] = w[0] * train['r_lgb'] + w[1] * train['r_cb'] + w[2] * train['r_xgb']
    return -ndcg_at_k_grouped(train, 'blend')

res = minimize(neg_ndcg, x0=best_w_grid, method='Nelder-Mead',
               options={'xatol': 0.001, 'fatol': 1e-7, 'maxiter': 500})
best_w_raw = np.abs(res.x)
best_w = tuple(best_w_raw / best_w_raw.sum())
best_score = -res.fun

print(f'\nNelder-Mead: LGB/CB/XGB = {[round(w, 4) for w in best_w]}')
print(f'OOF NDCG@5 (3-model blend): {best_score:.5f}')
print(f'Балл: {best_score * 100:.4f}')

In [ ]:
# === Submission v3 ===
test['pred_lgb'] = test_pred_lgb
test['pred_cb']  = test_pred_cb_avg
test['pred_xgb'] = test_pred_xgb
test['r_lgb'] = rank_within_group(test, 'pred_lgb')
test['r_cb']  = rank_within_group(test, 'pred_cb')
test['r_xgb'] = rank_within_group(test, 'pred_xgb')

w_lgb, w_cb, w_xgb = best_w
test['score'] = w_lgb * test['r_lgb'] + w_cb * test['r_cb'] + w_xgb * test['r_xgb']

submission = test[['request_id', 'variant_no', 'score']].copy()
submission.to_csv('submission_v3.csv', index=False, sep=';')

print('shape:', submission.shape)
print(submission.head(10))
assert submission['score'].isna().sum() == 0
print('✅ submission_v3.csv готов')

In [ ]:
# === Скачать submission ===
try:
    from google.colab import files
    files.download('submission_v3.csv')
except Exception as e:
    import subprocess
    out = subprocess.run(['curl', '-s', '-F', 'file=@submission_v3.csv', 'https://0x0.st'],
                          capture_output=True, text=True)
    print('Открой ссылку в браузере:', out.stdout.strip())

## Что делать если v3 не побил 1-е место

Если OOF v3 < 0.918 — есть ещё резервы:

1. **CatBoost QuerySoftMax** (4-я модель): теоретически идеален для one-positive
2. **LightGBM с `rank_xendcg`** objective вместо lambdarank (2-я LGB модель)
3. **Optuna 50 trials** на гиперпараметры LGB и CB
4. **Pseudo-labeling**: предсказать на test, взять confident, добавить в train
5. **LambdaMART stacking** через 2-й слой (paтtern из Expedia 5-place)

Каждое из них тестировать **отдельно**, сравнивать с v3 baseline.
